# Experimentos

Entrenamiento del agente de Q-learning tabular sobre el ambiente `DoorKey` con bola bloqueando la puerta. El agente debe coordinar subtareas (mover la bola, recoger la llave, abrir la puerta, llegar al objetivo) para obtener la recompensa terminal.

El flujo del notebook es:
1. Definir hiperparámetros.
2. Entrenar durante un número de episodios.
3. Guardar la Q-tabla resultante en `../results/`.
4. Evaluar el agente en modo greedy y reportar tasa de éxito y recompensa promedio.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))

from env import DoorKeyEnv
from agent import QLearningAgent

## Entrenamiento

Cada episodio se corta cuando el entorno marca `terminated`, `truncated`, o al alcanzar `max_steps`. `epsilon` decae geométricamente para pasar de exploración a explotación.

In [ ]:
N_EPISODES = 2000
MAX_STEPS = 300

env = DoorKeyEnv(seed=0)
agent = QLearningAgent(n_actions=env.n_actions, alpha=0.1, gamma=0.99,
                       epsilon=1.0, epsilon_min=0.05, epsilon_decay=0.995)

rewards_per_episode = []
for ep in range(N_EPISODES):
    state, _ = env.reset()
    total = 0.0
    for _ in range(MAX_STEPS):
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        agent.update(state, action, reward, next_state, done)
        state = next_state
        total += reward
        if done:
            break
    agent.decay_epsilon()
    rewards_per_episode.append(total)
    if (ep + 1) % 100 == 0:
        recent = sum(rewards_per_episode[-100:]) / 100
        print(f'ep {ep+1:4d}  eps={agent.epsilon:.3f}  avg_reward(last100)={recent:+.3f}')

## Guardar Q-tabla

In [ ]:
agent.save('../results/qtable_v1.pkl')
print('Q-tabla guardada. Tamaño (estados visitados):', len(agent.q))

## Curva de aprendizaje

In [ ]:
import matplotlib.pyplot as plt

window = 50
smoothed = [sum(rewards_per_episode[max(0, i-window):i+1]) / (min(i+1, window))
            for i in range(len(rewards_per_episode))]

plt.figure(figsize=(8, 4))
plt.plot(rewards_per_episode, alpha=0.3, label='por episodio')
plt.plot(smoothed, label=f'media móvil ({window})')
plt.xlabel('episodio')
plt.ylabel('recompensa total')
plt.title('Curva de aprendizaje')
plt.legend()
plt.tight_layout()
plt.show()

## Evaluación greedy

Cargamos la Q-tabla desde disco y ejecutamos el agente sin exploración para medir su desempeño.

In [ ]:
eval_agent = QLearningAgent.load('../results/qtable_v1.pkl')

N_EVAL = 20
successes = 0
eval_rewards = []
for _ in range(N_EVAL):
    state, _ = env.reset()
    total = 0.0
    for _ in range(MAX_STEPS):
        action = eval_agent.greedy_action(state)
        state, reward, terminated, truncated, _ = env.step(action)
        total += reward
        if terminated or truncated:
            if terminated and reward > 0:
                successes += 1
            break
    eval_rewards.append(total)

print(f'tasa de éxito: {successes}/{N_EVAL}')
print(f'recompensa promedio: {sum(eval_rewards)/N_EVAL:+.3f}')